# Claude Token & Cost Estimator

**Note:** This tool was originally created by Anthropic but I am trying to expand, improve, and keep it up to date with the latest models and their cost. If you want to contribute, let me know so we can collaborate.

**Added in this version — conversation context (section 4).** You can now paste the conversation so far alongside your prompt. The Messages API is stateless, so every prior turn is resent and re-billed on every request: a chat that feels like it costs one prompt's worth actually costs the whole transcript's worth, every turn. Section 7 splits your bill into "new prompt" versus "re-sending what you already sent", projects the next N turns (the cost grows quadratically), and shows what prompt caching would save.

Everything after this line, in this markdown file, was here originally.

-----------------------------

This notebook uses Anthropic's official **`/v1/messages/count_tokens`** endpoint to get an *exact* token count for a prompt against a specific Claude model, before you actually send it for generation. Counting tokens this way is free and doesn't count against your normal message rate limits.

It then estimates $ cost using current published per-token pricing (input cost is exact since it's based on the real count; output cost is an *estimate* since you choose how many output tokens to assume).

**Note on pricing:** rates below are current as of July 2026, but Anthropic can change them. If your numbers look off, check https://www.anthropic.com/pricing before trusting the cost estimate.

**Note on Sonnet 5 pricing:** it's on introductory pricing ($2/$10 per million tokens) through Aug 31, 2026, then standard pricing ($3/$15) from Sept 1, 2026. There's a toggle below for this.

## 1. Setup

In [1]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.7 MB/s eta 0:00:00


In [2]:
import getpass
import anthropic

api_key = getpass.getpass("Paste your Anthropic API key (input hidden): ")
client = anthropic.Anthropic(api_key=api_key)
print("Client ready.")

Paste your Anthropic API key (input hidden): ··········
Client ready.


## 2. Pricing table (per million tokens, USD)

Edit this dict if Anthropic updates pricing — treat it as the single source of truth for the rest of the notebook.

In [ ]:
# model_id: (input $ / MTok, output $ / MTok)
PRICING = {
    "claude-fable-5":(10.00, 50.00),
    "claude-opus-5":(5.00, 25.00),
    "claude-opus-4-8":(5.00, 25.00),
    "claude-sonnet-5":(3.00, 15.00),   # standard pricing, from Sept 1 2026
    "claude-sonnet-5-intro":(2.00, 10.00),   # introductory pricing through Aug 31 2026 (same model id, different rate)
    "claude-haiku-4-5":(1.00, 5.00),
}

MODEL_CHOICES = {
    "1": ("Claude Fable 5", "claude-fable-5"),
    "2": ("Claude Opus 5", "claude-opus-5"),
    "3": ("Claude Opus 4.8", "claude-opus-4-8"),
    "4": ("Claude Sonnet 5 (standard pricing, $3/$15)", "claude-sonnet-5"),
    "5": ("Claude Sonnet 5 (introductory pricing thru Aug 31 2026, $2/$10)", "claude-sonnet-5"),
    "6": ("Claude Haiku 4.5", "claude-haiku-4-5"),
}

# --- prompt caching rates, as multiples of the base input rate above ---
# Reads are dirt cheap; writes carry a premium that depends on the cache TTL.
CACHE_READ_MULTIPLIER = 0.10
CACHE_WRITE_MULTIPLIER = {"5m": 1.25, "1h": 2.00}
CACHE_TTL = "5m"   # "5m" or "1h"

# A prefix shorter than this silently does NOT cache — no error, no warning,
# you just never see a cache hit. Not monotonic across model generations.
MIN_CACHEABLE_TOKENS = {
    "claude-fable-5": 512,
    "claude-opus-5": 512,
    "claude-opus-4-8": 1024,
    "claude-sonnet-5": 1024,
    "claude-haiku-4-5": 4096,
}

# Fallback only. get_context_limit() below prefers a live Models API lookup so
# this table can't silently go stale the way a hardcoded one would.
CONTEXT_LIMIT_FALLBACK = {
    "claude-fable-5": 1_000_000,
    "claude-opus-5": 1_000_000,
    "claude-opus-4-8": 1_000_000,
    "claude-sonnet-5": 1_000_000,
    "claude-haiku-4-5": 200_000,
}

print("Pick a model:")
for k, (label, _) in MODEL_CHOICES.items():
    print(f"  {k}. {label}")

## 3. Choose model + enter your prompt

In [ ]:
choice = input("Enter the number for your model choice (1-6): ").strip()
label, model_id = MODEL_CHOICES.get(choice, MODEL_CHOICES["2"])
# Option 5 is the same model id as option 4, billed at the introductory rate.
pricing_key = "claude-sonnet-5-intro" if choice == "5" else model_id
print(f"Using: {label}  (model id sent to API: {model_id})")

In [ ]:
# Option A: type/paste your prompt directly here
prompt_text = """

"""

# Option B: uncomment to load a prompt from a local .txt file instead
# with open("my_prompt.txt", "r") as f:
#     prompt_text = f.read()

print(f"Prompt loaded ({len(prompt_text)} characters).")

Prompt loaded (18889 characters).


## 4. Optional: existing conversation context

If this prompt isn't the *first* message of a conversation, paste everything that
came before it here.

This matters more than it looks. The Messages API is **stateless** — there is no
conversation living on Anthropic's servers that you're topping up. Every prior
turn is **resent in full, and re-billed, on every single request**. So a chat that
feels like it costs one prompt's worth actually costs the whole transcript's worth,
every turn, and grows *quadratically* as it goes.

Leave it blank and the notebook behaves exactly as it did before.

**Paste format:** by default the context is treated as one opaque blob, which is
accurate to within a handful of tokens. If your transcript uses `User:` /
`Assistant:` turn markers and you want the exact per-message accounting the API
actually bills, set `PARSE_CONTEXT_TURNS = True`.

In [ ]:
# Option A: paste the conversation so far here. Leave blank for a fresh prompt.
context_text = """

"""

# Option B: uncomment to load the context from a local .txt file instead
# with open("my_context.txt", "r") as f:
#     context_text = f.read()

PARSE_CONTEXT_TURNS = False  # True if your paste uses User:/Assistant: markers

if context_text.strip():
    print(f"Context loaded ({len(context_text)} characters).")
else:
    print("No context — pricing this prompt as a standalone, first-turn request.")

In [ ]:
import re

# A turn marker at the start of a line: "User:", "Human:", "Assistant:", "Claude:"
_TURN_MARKER = re.compile(
    r"^[ \t]*(User|Human|Assistant|Claude)[ \t]*:[ \t]*",
    re.IGNORECASE | re.MULTILINE,
)
_ROLE_FOR = {"user": "user", "human": "user", "assistant": "assistant", "claude": "assistant"}


def parse_context_turns(text):
    """Split a pasted transcript into API messages on User:/Assistant: markers.

    Returns [] when the text contains no turn markers at all, so the caller can
    fall back to treating the paste as a single blob.
    """
    marks = list(_TURN_MARKER.finditer(text))
    if not marks:
        return []

    turns = []
    for i, m in enumerate(marks):
        role = _ROLE_FOR[m.group(1).lower()]
        end = marks[i + 1].start() if i + 1 < len(marks) else len(text)
        content = text[m.end():end].strip()
        if not content:
            continue
        # The API rejects two messages in a row with the same role, so fold them.
        if turns and turns[-1]["role"] == role:
            turns[-1] = {"role": role, "content": turns[-1]["content"] + "\n\n" + content}
        else:
            turns.append({"role": role, "content": content})

    # messages[] must open with a user turn. Dropping content would undercount
    # the estimate, so say so out loud rather than swallowing it.
    if turns and turns[0]["role"] != "user":
        dropped = turns.pop(0)
        print(
            f"  ! Transcript starts with an assistant turn; dropped "
            f"{len(dropped['content'])} characters (messages must begin with a "
            f"user turn). Use blob mode instead if that text should be counted."
        )
    return turns


def build_messages(context_text, prompt_text, parse_turns=False):
    """Assemble the messages list exactly as it would be sent for real."""
    ctx = (context_text or "").strip()
    prompt = (prompt_text or "").strip()

    if not ctx:
        return [{"role": "user", "content": prompt}]

    if parse_turns:
        turns = parse_context_turns(ctx)
        if turns:
            if not prompt:
                return turns
            if turns[-1]["role"] == "user":
                # Don't emit two user turns back to back — fold the prompt in.
                merged = {"role": "user", "content": turns[-1]["content"] + "\n\n" + prompt}
                return turns[:-1] + [merged]
            return turns + [{"role": "user", "content": prompt}]
        print("  ! No User:/Assistant: markers found — falling back to blob mode.")

    # Blob mode: the whole context plus the new prompt as one user turn.
    return [{"role": "user", "content": f"{ctx}\n\n{prompt}" if prompt else ctx}]


print("Context helpers ready.")

## 5. Optional: system prompt
System prompt tokens count too — include one here if your real request will use one. Leave blank if not.

In [ ]:
system_prompt = ""  # leave as empty string if you don't use one

## 6. Count tokens (exact, via the API — free, no generation happens)

Counts the request as it would actually be sent: system prompt + context + new prompt.

The context and prompt are counted separately so section 7 can show you how much of
your bill is the new prompt and how much is re-sending what you already sent. The
prompt figure is a *marginal* count — total minus context — which is the number that
actually matters and sidesteps per-message framing overhead.

In [ ]:
def count(messages, system=None):
    """Exact token count for a request, via the free count_tokens endpoint."""
    kwargs = {"model": model_id, "messages": messages}
    if system and system.strip():
        kwargs["system"] = system
    return client.messages.count_tokens(**kwargs).input_tokens


def get_context_limit(model_id):
    """The model's context window, live from the Models API where possible.

    max_input_tokens is the context window (there is no context_window field).
    Falls back to the hardcoded table if the lookup fails or the field is absent.
    """
    try:
        limit = getattr(client.models.retrieve(model_id), "max_input_tokens", None)
        if limit:
            return limit
    except Exception as e:
        print(f"  ! Models API lookup failed ({type(e).__name__}); using fallback table.")
    return CONTEXT_LIMIT_FALLBACK.get(model_id)


messages = build_messages(context_text, prompt_text, PARSE_CONTEXT_TURNS)
total_input_tokens = count(messages, system_prompt)

# Context alone, so section 7 can split the bill. Skipped when there's no context.
if context_text.strip():
    context_tokens = count(build_messages(context_text, "", PARSE_CONTEXT_TURNS), system_prompt)
else:
    context_tokens = 0

new_prompt_tokens = total_input_tokens - context_tokens
context_limit = get_context_limit(model_id)

print(f"Model: {label}")
print(f"  context tokens:     {context_tokens:>10,}")
print(f"  new prompt tokens:  {new_prompt_tokens:>10,}  (marginal)")
print(f"  total input tokens: {total_input_tokens:>10,}")
if context_limit:
    pct = total_input_tokens / context_limit * 100
    print(f"  context window:     {total_input_tokens:,} / {context_limit:,}  ({pct:.1f}% used)")

## 7. Estimate cost

Input cost is exact (based on the real counts above). Output cost is a guess — enter roughly how many tokens you expect Claude to generate in response (a rough rule of thumb is ~0.75 tokens per word for plain English prose; more for code).

Three views, all in dollars:

**(a) This turn** — split into what you're paying to re-send versus what you're paying for the new prompt.

**(b) The next N turns** — because context is resent every request, a conversation's cost grows *quadratically*, not linearly. Turn 20 pays for turns 1–19 all over again.

**(c) With prompt caching** — the same N turns if you cache the stable prefix. Cache reads cost 0.1× and writes 1.25×, so on a long conversation with a big context this is usually the difference between a rounding error and a real bill.

In [ ]:
estimated_output_tokens = 500  # <-- change this to whatever you expect the response length to be
turns_to_project = 10          # <-- how many more turns to project in views (b) and (c)

input_rate, output_rate = PRICING[pricing_key]


def money(x):
    """Dollar formatting that stays readable across six orders of magnitude."""
    if x >= 1:
        return f"${x:,.2f}"
    if x >= 0.01:
        return f"${x:.4f}"
    return f"${x:.6f}"


def input_tokens_at_turn(i):
    """Prompt size on the i-th future turn (1-indexed), as the context grows.

    Each turn appends your prompt and Claude's reply, so turn i resends the
    original context plus everything accumulated before it.
    """
    return context_tokens + i * new_prompt_tokens + (i - 1) * estimated_output_tokens


# ---------- (a) this turn ----------
context_cost = (context_tokens / 1_000_000) * input_rate
prompt_cost = (new_prompt_tokens / 1_000_000) * input_rate
output_cost = (estimated_output_tokens / 1_000_000) * output_rate
turn_cost = context_cost + prompt_cost + output_cost

print(f"Model: {label}")
print()
print("(a) This turn")
print(f"      {'':<22}{'tokens':>12}{'cost ($)':>14}")
print(f"      {'re-sent context':<22}{context_tokens:>12,}{context_cost:>14,.6f}")
print(f"      {'new prompt':<22}{new_prompt_tokens:>12,}{prompt_cost:>14,.6f}")
print(f"      {'output (assumed)':<22}{estimated_output_tokens:>12,}{output_cost:>14,.6f}")
print(f"      {'-' * 48}")
print(f"      {'this turn':<22}{total_input_tokens:>12,}{turn_cost:>14,.6f}")
if context_tokens:
    print(f"      -> {context_cost / turn_cost * 100:.1f}% of this turn is re-sending context you already paid for")

# ---------- (b) the next N turns, uncached ----------
uncached_total = 0.0
overflow_turn = None
for i in range(1, turns_to_project + 1):
    tokens_in = input_tokens_at_turn(i)
    if context_limit and overflow_turn is None and tokens_in > context_limit:
        overflow_turn = i
    uncached_total += (tokens_in / 1_000_000) * input_rate
    uncached_total += (estimated_output_tokens / 1_000_000) * output_rate

growth_per_turn = new_prompt_tokens + estimated_output_tokens
print()
print(f"(b) Next {turns_to_project} turns, no caching")
print(f"      context grows ~{growth_per_turn:,} tokens/turn, ending at ~{input_tokens_at_turn(turns_to_project):,}")
print(f"      total: {money(uncached_total)}   (vs {money(turn_cost * turns_to_project)} if cost stayed flat)")
if overflow_turn:
    print(f"      ! exceeds the {context_limit:,}-token window at turn {overflow_turn}")

# ---------- (c) the same N turns, with prompt caching ----------
write_multiplier = CACHE_WRITE_MULTIPLIER[CACHE_TTL]
min_cacheable = MIN_CACHEABLE_TOKENS.get(model_id, 1024)

cached_total = 0.0
for i in range(1, turns_to_project + 1):
    tokens_in = input_tokens_at_turn(i)
    # Everything cached by the previous turn is read back cheaply; only this
    # turn's new material is written at the cache-write premium.
    read = input_tokens_at_turn(i - 1) if i > 1 else 0
    write = tokens_in - read
    cached_total += (read * CACHE_READ_MULTIPLIER + write * write_multiplier) / 1_000_000 * input_rate
    cached_total += (estimated_output_tokens / 1_000_000) * output_rate

print()
print(f"(c) Next {turns_to_project} turns, with prompt caching ({CACHE_TTL} TTL)")
if input_tokens_at_turn(1) < min_cacheable:
    print(f"      ! prefix is {input_tokens_at_turn(1):,} tokens, below this model's "
          f"{min_cacheable:,}-token minimum — caching would silently never engage")
else:
    print(f"      total: {money(cached_total)}")
    print(f"      saves: {money(uncached_total - cached_total)} "
          f"({(1 - cached_total / uncached_total) * 100:.1f}% off the uncached run)")

## 8. (Optional) Batch-check multiple prompts at once

Useful if you want to estimate cost across several prompts before running a batch (e.g. many podcast script drafts, citation checks, etc.).

Each prompt is priced against the same `context_text` from section 4, so this also answers "I have one big context and several candidate prompts — what does each cost?" Set `APPLY_CONTEXT_TO_BATCH = False` to price them standalone instead.

In [ ]:
prompts = [
    "First prompt text goes here.",
    "Second prompt text goes here.",
    # add as many as you want
]

assumed_output_tokens_each = 500
APPLY_CONTEXT_TO_BATCH = True  # False to price each prompt with no context

# Computed here rather than inherited from section 7, so this cell stands alone.
input_rate, output_rate = PRICING[pricing_key]
batch_context = context_text if APPLY_CONTEXT_TO_BATCH else ""

total_input_tokens_batch = 0
print(f"{'#':<4}{'input tokens':<15}{'est. cost':<12}")
for i, p in enumerate(prompts, 1):
    r_tokens = count(build_messages(batch_context, p, PARSE_CONTEXT_TURNS), system_prompt)
    total_input_tokens_batch += r_tokens
    cost = (r_tokens / 1_000_000) * input_rate + (assumed_output_tokens_each / 1_000_000) * output_rate
    print(f"{i:<4}{r_tokens:<15,}${cost:.6f}")

grand_total_cost = (
    (total_input_tokens_batch / 1_000_000) * input_rate
    + (len(prompts) * assumed_output_tokens_each / 1_000_000) * output_rate
)
print(f"\nTotal input tokens across {len(prompts)} prompts: {total_input_tokens_batch:,}")
print(f"Total estimated cost: ${grand_total_cost:.6f}")

if APPLY_CONTEXT_TO_BATCH and context_text.strip():
    shared = count(build_messages(context_text, "", PARSE_CONTEXT_TURNS), system_prompt)
    shared_cost = (shared * len(prompts) / 1_000_000) * input_rate
    print(f"\nOf that, {money(shared_cost)} is the same {shared:,}-token context sent {len(prompts)} times.")
    print("Prompt caching would collapse that to roughly one write plus cheap reads — see section 7(c).")